# 用PROC GLMSELECT寻找需求驱动因素：逐步选择、LASSO与验证前向选择

## 摘要

一支零售分析团队使用 **PROC GLMSELECT** 来发现哪些营销和定价杠杆真正影响每周单位销量，从而把真正的需求驱动因素与噪声区分开来。以SBC评分的逐步选择、带5折交叉验证的LASSO，以及经留出集验证的前向搜索，三种方法都恢复出**同一组真实驱动因素**——自身价格、竞争对手价格、广告支出、电子邮件量、促销、假日、东北（Northeast）区域以及在线（Online）渠道——并且四个人为设置的干扰变量（`temp_f`、`noise1`、`noise2`、`noise3`）全部被自动剔除。

三种方法在效应量级上也高度一致：每种方法估计的自身价格效应都接近**每美元-28个单位**，竞争对手价格效应接近**+14**——这正是数据生成方程内置的替代关系符号。唯一的分歧出现在边缘处——经交叉验证的LASSO额外保留了一个很小、统计上可忽略的`Region=Midwest`对比项（估计值-8.3，标准误8.3），而逐步选择和前向选择都将其剔除。一份能经受住三种不同选择方法检验的驱动因素清单，远比只针对单一规则调优得到的清单更可信。

## 数据来源

本笔记本中的所有数据均为**合成数据**，在代码中内联生成（无外部文件，随机数种子`20260531`）。它模拟了一家消费品零售商一个销售季度的门店周需求面板数据。

| 数据集 | 行数 | 粒度 | 关键变量 |
|---------|------|-------|---------------|
| `demand` | 100 | 门店-周 | `units`（响应变量：每周销量）；`price`、`competitor`（自身与对手货架价格）；`adspend`、`email`（媒介投放力度）；`promo`、`holiday`（事件标志）；`region`、`channel`（CLASS效应）；`temp_f`、`noise1`-`noise3`（干扰/无关预测变量）|

`units`由一个已知的线性方程构建而成，因此“正确”的驱动因素集合是可验证的；`temp_f`和三个`noise`列不携带任何真实信号，用来检验每种选择方法能否将其剔除。

# 用PROC GLMSELECT寻找需求驱动因素

一位品类经理面对着数十个每周销量的候选解释变量：自身价格、竞争对手价格、广告支出、电子邮件量、促销、假日、门店区域、销售渠道，甚至天气。把它们全部塞进一个回归模型会导致过拟合和系数不稳定。**PROC GLMSELECT**自动搜索简约模型，支持逐步、LASSO、弹性网和最小角回归选择，并内置交叉验证和留出集划分。

在本笔记本中我们将：

1. 生成一份具有*已知*真实驱动因素集合（外加人为设置的干扰变量）的逼真合成门店周需求面板数据。
2. 运行以SBC评分的**逐步选择**。
3. 运行带5折交叉验证的**LASSO**。
4. 运行在30%留出集上验证的**前向选择**。

一个好的选择方法应当能恢复出真实驱动因素并剔除噪声——我们拭目以待。

## 1. 生成合成需求面板数据

响应变量`units`由一个显式的线性方程构造而成，因此我们知道真实情况：价格和竞争对手价格、广告支出、电子邮件量、促销和假日标志，以及区域和渠道效应都起作用。变量`temp_f`、`noise1`、`noise2`和`noise3`是纯粹的干扰变量，与销量没有真实关系。`call streaminit`随机数种子使数据可复现。

In [1]:
/* ---------------------------------------------------------------
   面向消费品零售商的合成门店周需求面板数据。
   units（销量）遵循已知方程；temp_f 和 noise1-3 为干扰变量。
   --------------------------------------------------------------- */
数据 demand;
    调用 streaminit(20260531);
    长度 region $9 channel $8 promo $3;
    循环 store_week = 1 到 100;
        /* 区域分布 */
        u1 = rand('uniform');
        如果 u1 < 0.34 那么 region = 'Northeast';
        否则 如果 u1 < 0.67 那么 region = 'Midwest';
        否则 region = 'South';

        /* 销售渠道 */
        如果 rand('uniform') < 0.45 那么 channel = 'Store';
        否则 channel = 'Online';

        /* 定价与媒介投放驱动因素 */
        price      = round(8  + rand('uniform') * 12, 0.01);
        competitor = round(8  + rand('uniform') * 12, 0.01);
        adspend    = round(rand('gamma', 2) * 500, 1);
        email      = round(rand('uniform') * 100, 1);

        /* 事件标志和一个无关的天气干扰变量 */
        temp_f     = round(40 + rand('uniform') * 50, 0.1);
        holiday    = (rand('uniform') < 0.12);
        如果 rand('uniform') < 0.40 那么 promo = 'Yes';
        否则 promo = 'No';

        /* 纯噪声预测变量（无真实影响） */
        noise1 = rand('normal');
        noise2 = rand('normal');
        noise3 = rand('normal');

        /* 每周销量由已知结构方程生成 */
        units = 900
              - 28   * price
              + 14   * competitor
              + 0.06 * adspend
              + 1.2  * email
              + 55   * (promo = 'Yes')
              + 70   * holiday
              + 40   * (region = 'Northeast')
              + 25   * (channel = 'Online')
              + 30   * rand('normal');
        如果 units < 0 那么 units = 0;
        输出;
    结束;
    保留 region channel promo price competitor adspend email temp_f
         holiday noise1 noise2 noise3 units;
运行;


NOTE: DATA demand


NOTE: Wrote demand (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.03 seconds
  cpu   0.03 seconds


## 2. 数据概况

在建模之前，先确认响应变量和主要连续候选变量的数值范围是否合理。

In [2]:
过程 均值 数据=demand n mean std MIN MAX maxdec=1;
    变量 units price competitor adspend email;
    标签 units="销量" price="自身价格" competitor="竞争对手价格"
          adspend="广告支出" email="电子邮件量";
    标题 "每周需求与候选驱动因素";
运行;

                                                      每周需求与候选驱动因素                                                       

                                                  The MEANS Procedure

 Variable    Label                      N        Mean     Std Dev     Minimum     Maximum
 ----------------------------------------------------------------------------------------
 units       销量                       100       874.8       136.3       508.6      1162.3
 price       自身价格                     100        14.0         3.4         8.0        19.9
 competitor  竞争对手价格                   100        13.8         3.4         8.1        19.9
 adspend     广告支出                     100       990.6       726.9        50.0      3358.0
 email       电子邮件量                    100        45.5        26.4         0.0        99.0
 ----------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 3. 以SBC评分的逐步选择

逐步选择交替地添加和移除效应，这里以**施瓦茨贝叶斯准则（SBC）**同时作为入选检验标准（`select=sbc`）和最终模型选择标准（`choose=sbc`）。SBC比AIC对模型复杂度的惩罚更重，因而偏好更精简的模型。

关键语句和选项：

- `CLASS region channel promo / param=reference`以参考编码方式声明分类预测变量。
- `selection=stepwise(select=sbc choose=sbc)`驱动搜索过程。
- `details=summary`打印逐步选择摘要；`stb`添加标准化系数，便于比较不同量纲的效应。
- `output out=demand_scored p=predicted r=residual`保存拟合值和残差，供后续评分使用。

将**逐步选择摘要（Stepwise Selection Summary）**读作搜索轨迹：它从完整的12个效应模型开始，逐一*移除*效应，依次剔除`noise1`、`noise2`、`temp_f`、`Region=Midwest`对比项和`noise3`，因为每次移除都降低了SBC。**参数估计（Parameter Estimates）**表中保留下来的就是最终选定的模型。

In [3]:
过程 glmselect 数据=demand seed=20260531;
    分类 region channel promo / PARAM=reference;
    模型 units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=stepwise(选择=sbc choose=sbc)
          details=summary stb;
    标签 units="销量" region="区域" channel="渠道" promo="促销"
          price="自身价格" competitor="竞争对手价格" adspend="广告支出"
          email="电子邮件量" temp_f="气温（华氏度）" holiday="假日"
          noise1="噪声变量1" noise2="噪声变量2" noise3="噪声变量3";
    输出 out=demand_scored p=predicted r=residual;
    标题 "需求驱动因素的逐步选择（SBC）";
运行;

                                                      每周需求与候选驱动因素                                                       

The GLMSELECT Procedure


Dependent Variable: UNITS 销量


Number of Observations Used: 100

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                    Stepwise Selection Summary                                                    

    Step    Action                 Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)
--------  --------  ---------------------  -----------------  --------  --------  --------  --------  --------  --------  --------
       1   Removed                  噪声变量1                 12    0.9507    0.9439  707.0094  711.2420  713.1843  740.8766   12.0


NOTE: PROC GLMSELECT data=demand

NOTE: The data set WORK.DEMAND_SCORED has 100 observations and 15 variables.
NOTE: PROC GLMSELECT statement used.


## 4. 带交叉验证的LASSO

LASSO将系数向零收缩，同时完成选择和正则化。我们不采用固定的停止准则，而是让**5折交叉验证**在LASSO路径上选出样本外误差最小的那一步。

- `selection=lasso(choose=cv stop=none)`追踪完整的LASSO路径并选取CV最优的步骤。
- `cvmethod=random(5)`将观测值随机分配到5个折。

**LASSO选择摘要（LASSO Selection Summary）**展示了随着惩罚项放松，各效应进入模型的顺序：`price`最先进入，然后是`adspend`、`competitor`、东北（Northeast）区域、`email`、`promo`，最后是`holiday`——所有七个真实信号都先于任何干扰变量进入模型。交叉验证随后选择样本外PRESS最小的那一步，最终的**参数估计（Parameter Estimates）**表恰好保留了真正的驱动因素（外加一个很小的`Region=Midwest`项），而排除了`temp_f`、`noise1`、`noise2`和`noise3`，它们只在路径的最末端才进入模型。

In [4]:
过程 glmselect 数据=demand seed=20260531;
    分类 region channel promo / PARAM=reference;
    模型 units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=lasso(choose=cv 停止=none)
          cvmethod=RANDOM(5);
    标签 units="销量" region="区域" channel="渠道" promo="促销"
          price="自身价格" competitor="竞争对手价格" adspend="广告支出"
          email="电子邮件量" temp_f="气温（华氏度）" holiday="假日"
          noise1="噪声变量1" noise2="噪声变量2" noise3="噪声变量3";
    标题 "5折交叉验证LASSO";
运行;

                                                      每周需求与候选驱动因素                                                       

The GLMSELECT Procedure


Dependent Variable: UNITS 销量


Number of Observations Used: 100

  Cross Validation Information   

                   Item     Value
-----------------------  --------
Cross Validation Method    Random
        Number of Folds         5

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                            LASSO Selection Summary                                                            

    Step    Action                 Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        PRESS
--------  --------  ---------------------  -----------------


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 5. 在留出集上验证的前向选择

第三种互补策略：通过**前向选择**（效应只能进入，不能离开）构建模型，但在独立留出样本上表现最佳的地方停止——直接防范过拟合。

- `partition fraction(validate=0.30)`随机保留30%的行用于验证，剩下70个训练观测和30个验证观测。
- `selection=forward(select=aic choose=validate)`按AIC添加效应，但按验证样本误差选择最终模型。

**分区比例（Partition Fractions）**表确认了70/30的划分。前向选择随后不断加入效应，直到验证误差不再改善；最终**参数估计（Parameter Estimates）**表中的八个效应恰好就是真实驱动因素，四个干扰变量始终未被纳入。当基于不同原理构建的三种方法都得到同样的驱动因素时，这个模型远比任何单一选择规则的产物更可能是真实的。

In [5]:
过程 glmselect 数据=demand seed=20260531;
    分类 region channel promo / PARAM=reference;
    模型 units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=forward(选择=aic choose=validate);
    标签 units="销量" region="区域" channel="渠道" promo="促销"
          price="自身价格" competitor="竞争对手价格" adspend="广告支出"
          email="电子邮件量" temp_f="气温（华氏度）" holiday="假日"
          noise1="噪声变量1" noise2="噪声变量2" noise3="噪声变量3";
    partition FRACTION(validate=0.30);
    标题 "基于30%留出验证集的前向选择";
运行;

                                                      每周需求与候选驱动因素                                                       

The GLMSELECT Procedure


Dependent Variable: UNITS 销量


Number of Observations Used: 70               
Number of Observations Used for Validation: 30

Partition Fractions 

      Role  Fraction
----------  --------
  Training    0.7000
Validation    0.3000
   Testing    0.0000

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                         Forward Selection Summary                                                          

    Step    Action              Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        PRESS
--------  --------  ------------------  ---------


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 6. 结果解读

三种选择策略都恢复出**同一组真实需求驱动因素**并剔除了所有干扰变量。直接读取**参数估计（Parameter Estimates）**表：

- **价格**是主导驱动因素。在逐步模型中它的标准化系数为**-0.70**，绝对值远大于其他系数，原始斜率落在**-27.8**（逐步和LASSO）到**-28.6**（前向）单位/美元之间——几乎正好等于数据生成时使用的-28。**竞争对手价格**在三个模型中都以大约**+14.4**的方向反向影响销量，符合品类经理预期的替代行为。
- **广告支出**（约每美元+0.062个单位）和**电子邮件量**（约每次发送+1.2个单位）都能提升销量，量化了媒介响应。
- **促销**和**假日**带来最大的事件效应：`Promo=No`对比项相对于促销周约为**-51到-57**，假日提升幅度约为**+66到+76**个单位。**东北（Northeast）区域**（约+49到+55）和**在线（Online）渠道**（约+24到+32）带有正向的基线效应。
- 关键的是，每一个干扰变量——`temp_f`、`noise1`、`noise2`、`noise3`——都被逐步选择和前向选择**剔除**，并且被交叉验证选定的LASSO模型排除。每种方法都恢复了结构信号并忽略了噪声。

三种方法并非逐字节完全一致：逐步选择（SBC）和经留出集验证的前向搜索都选定了同样的八个效应，而经交叉验证的LASSO额外保留了一个很小的`Region=Midwest`对比项（-8.3，标准误8.3）——这是噪声水平的差异，而非实质性分歧。一份能同时经受逐步SBC、交叉验证LASSO和留出集验证前向选择检验的驱动因素清单，才是真正的收获：一个能经受三种不同选择方法检验的发现，远比只针对单一准则调优得到的发现更可信。借助`OUTPUT OUT=demand_scored`捕获拟合值和残差，同样的工作流程可以自然地扩展到对下一季度计划的价格和促销日历进行评分。